## Microprice

$$
m_t = \frac{a_t V_t^b + b_t V_t^a}{V_t^b + V_t^a}
$$

where:
- $b_t$ — best bid
- $a_t$ — best ask
- $V_t^b$ — Volume on best bid
- $V_t^a$ — Volume on best ask

In [12]:
def microprice(best_bid: float, best_ask: float, bid_size: float, ask_size: float) -> float:
    bid_ask = bid_size + ask_size
    return (best_ask * bid_size + best_bid * ask_size) / bid_ask

In [13]:
microprice(70001.0, 70002.0, bid_size=100000, ask_size=50000)

70001.66666666667

## Reservation price for infinite-horizon horizon

$$
r_t = m_t - \theta q_t
$$

where:
- $m_t$ — microprice
- $q_t$ — inventory
- $\theta$ — inventory penalty coefficient

Reservation price is actualy a price according to which MM is willing to place bid and ask orders based on amount of inventory and self-defined risk

In [32]:
def reservation_price(microprice_t: float, inventory: float, theta: float) -> float:
    if inventory == 0:
        o_inventory = microprice_t - 2.0
        print(o_inventory)
    else:
        print(microprice_t - theta * inventory)
    return

In [34]:
reservation_price(microprice_t=70001.67, inventory=20000, theta=0.0006)

69989.67


## Half-spread

We define the **half-spread** $\delta_t$.

The bid and ask quotes are then given by

$$
p_t^{bid} = r_t - \delta_t
$$

$$
p_t^{ask} = r_t + \delta_t
$$

where:
- $r_t$ is the reservation price
- $\delta_t$ is the half-spread

So the reservation price is the **center** of our quotes, and the half-spread determines how far from that center we place the bid and the ask.

The total spread is therefore

$$
p_t^{ask} - p_t^{bid} = 2\delta_t
$$

For a practical infinite-horizon MM model, we can define the half-spread as

$$
\delta_t = \delta_0 + c_\sigma \sigma_t + c_q |q_t|
$$

where:
- $\delta_0$ is the base half-spread
- $\sigma_t$ is the short-term volatility estimate
- $q_t$ is the current inventory
- $c_\sigma$ controls how much volatility widens the spread
- $c_q$ controls how much inventory widens the spread

So in the model:
- the reservation price manages **directional skew**
- the half-spread manages **quote width**

In [35]:
def compute_quotes(base_bid: float, base_ask: float, inventory: float, sigma_t: float, c_sigma: float, c_q: float, r_t: float):
    base_half_spread = (base_ask - base_bid) / 2
    print(base_half_spread)
    delta_t = base_half_spread + sigma_t*c_sigma + c_q*abs(inventory)
    p_bid_t = r_t - delta_t
    p_ask_t = r_t + delta_t
    return delta_t, p_bid_t, p_ask_t

In [36]:
compute_quotes(70001.0, 70002.0, inventory=20000, sigma_t=2, c_sigma=0.5, c_q=0.00006, r_t=69989.666 )

0.5


(2.7, 69986.966, 69992.366)

## Amount Levels for microprice 


### OB volatility index
$$
V_{t OBindx} = ln \left(\frac{(V_t^{1,a} + V_t^{2,a} + V_t^{3,a}) - (V_t^{1,b} + V_t^{2,b} + V_t^{3,b})}{(V_{t-1}^{1,a} + V_{t-1}^{2,a} + V_{t-1}^{3,a}) - (V_{t-1}^{1,b} + V_{t-1}^{2,b} + V_{t-1}^{3,b})}\right)
$$

Where:
- $V_{t-1}^{1,a}$ - Ask volume L1 on previous tick
- $V_{t-1}^{1,b}$ - Bid volume L1 on previous tick


$$
\bar{V}_{OBindx,\,5} = \frac{1}{5}\sum_{i=0}^{4} V_{OBindx,\,t-i}
$$


## how many OB levels
$$
L_t = f\!\left(\bar{V}_{OBindx,t}^{(5)}\right)
$$

## Microprice OB Vol index (dynamic depth)
$$
m_t^{(L_t)} =
\frac{
\sum_{k=1}^{L_t} \left(a_t^k V_t^{k,b} + b_t^k V_t^{k,a}\right)
}{
\sum_{k=1}^{L_t} \left(V_t^{k,b} + V_t^{k,a}\right)
}
$$

### If L_t = 3
$$
m_t^{(3)} =
\frac{
(a_t^1 V_t^{1,b} + b_t^1 V_t^{1,a})
+
(a_t^2 V_t^{2,b} + b_t^2 V_t^{2,a})
+
(a_t^3 V_t^{3,b} + b_t^3 V_t^{3,a})
}{
V_t^{1,b} + V_t^{1,a} + V_t^{2,b} + V_t^{2,a} + V_t^{3,b} + V_t^{3,a}
}
$$

In [144]:
class between:
    def __init__(self, idx:float):
        self.idx = idx
    def calc_if_between(self):
        if self.idx > 0.6 and self.idx <=0.8:
            result = 4
        elif self.idx <= 0.6:
            result = 3
        else:
            result = 5
        return result

def ob_index_to_lvl(v_ob_indx_1: float,v_ob_indx_2: float,v_ob_indx_3: float,v_ob_indx_4: float,v_ob_indx_5: float):
    
    ob_idx = [v_ob_indx_1, v_ob_indx_2, v_ob_indx_3, v_ob_indx_4, v_ob_indx_5]
    levels = []

    for i in ob_idx:
        if abs(i) > 1:
            v_ob_levels = 5
        else:
            v_ob_levels = between(abs(i)).calc_if_between()
        levels.append(v_ob_levels)

    lvl_sum = 0
    for level in levels:
        lvl_sum += level

    lvl_amount = int(round(lvl_sum / len(levels)))

    return  lvl_amount


## Imbalance

$$
V_B = V_t^{1,b} + V_t^{2,b} + V_t^{3,b}
$$

$$
V_A = V_t^{1,a} + V_t^{2,a} + V_t^{3,a}
$$


$$
V_{imb} = \frac{V_B - V_A}{V_B + V_A} 
$$

## Microprice 3 levels

$$
m_t = \frac{a_t^1 V_t^{1,b} + b_t^1 V_t^{1,a} + a_t^2 V_t^{2,b} + b_t^2 V_t^{2,a} + a_t^3 V_t^{3,b} + b_t^3 V_t^{3,a}}{V_t^{1,b} + V_t^{1,a} + V_t^{2,b} + V_t^{2,a} + V_t^{3,b} + V_t^{3,a}}
$$


## Optimal price

$$
P_{opt} = m_tV_{imb}/100)
$$


Where:
- $b_t^1$ - bid on L1
- $a_t^1$ - ask on L1
- $V_t^{1,b}$ - Bid volume L1
- $V_t^{1,a}$ - Ask volume L1


In [122]:
import math

def microprice(b1_t: float, a1_t: float, v_1b: float, v_1a: float, b2_t: float, a2_t: float, v_2b: float, v_2a: float, b3_t: float, 
               a3_t: float, v_3b: float, v_3a: float, v_t1b: float, v_t1a: float, v_t2b: float, v_t2a: float, v_t3b: float, v_t3a: float ) -> float:
    vol_sum = v_1b + v_1a + v_2b + v_2a + v_3b + v_3a
    microprice_l = (a1_t * v_1b + b1_t * v_1a + a2_t * v_2b + b2_t * v_2a + a3_t * v_3b + b3_t * v_3a) / vol_sum
    ob_vol_index = (math.log(((v_1a + v_2a + v_3a) - (v_1b + v_2b + v_3b)) / ((v_t1a + v_t2a + v_t3a) - (v_t1b + v_t2b + v_t3b)))) / 100
    V_B = v_1b + v_2b + v_3b
    V_A = v_1a + v_2a + v_3a
    imbalance = ((V_B - V_A) / (V_B + V_A)) / 10
    opt_price = microprice_l * (1. + imbalance)
    return microprice_l, ob_vol_index, imbalance, opt_price

In [123]:
microprice(b1_t=0.002661, a1_t=0.002662, v_1b=310821.0, v_1a=286383.0, b2_t=0.002660, a2_t=0.002662, v_2b=791404.0, v_2a=1092230.0, 
           b3_t=0.002659, a3_t=0.002663, v_3b=1454561.0, v_3a=1362703.0,  v_t1b=110248.0, v_t1a=1319334.0, v_t2b=775953.0,
           v_t2a= 1706612.0, v_t3b=2149935.0, v_t3a=1247695.0)

(0.0026610365623387397,
 -0.019030404829503366,
 -0.0034829454019571536,
 0.0026517683172795022)